# LSTM Neural Baseline (GloVe + BiLSTM)
Compatible with your baseline pipeline:
- Loads fixed splits from `artifacts/splits/`
- Trains **binary** classifier (NEG vs POS)
- Applies **Neutral post-hoc thresholding** at inference
- Exports predictions to `artifacts/predictions/`.

## Prerequisite: Download GloVe once
Download `glove.6B.zip` from Stanford and extract `glove.6B.300d.txt`.
Place it at: `./glove/glove.6B.300d.txt` (relative to this notebook).
Source: https://nlp.stanford.edu/projects/glove/

In [ ]:
# === 1) Setup & Paths ===
import os, json
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report, confusion_matrix

# Tokenization / padding
from keras_preprocessing.text import Tokenizer
from keras_preprocessing.sequence import pad_sequences

BASE_DIR = Path(".")
ARTIFACT_DIR = BASE_DIR / "artifacts"

SPLIT_DIR = ARTIFACT_DIR / "splits"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PRED_DIR = ARTIFACT_DIR / "predictions"
PRED_DIR.mkdir(exist_ok=True)

MODEL_DIR = ARTIFACT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

CONFIG_DIR = ARTIFACT_DIR / "config"
CONFIG_DIR.mkdir(exist_ok=True)

TRAIN_PATH = SPLIT_DIR / "train_60.csv"
VAL_PATH   = SPLIT_DIR / "val_10.csv"
TEST_PATH  = SPLIT_DIR / "test_30.csv"

# Put glove.6B.300d.txt here:
GLOVE_PATH = BASE_DIR / "glove" / "glove.6B.300d.txt"

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Hyperparameters
RANDOM_STATE = 42
MAX_VOCAB = 50_000
MAX_LEN = 50
EMBED_DIM = 300
HIDDEN_DIM = 128
BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-3

# Neutral thresholds (post-hoc)
NEG_TH = 0.40
POS_TH = 0.60


Using device: cpu


In [15]:
# === 2) Load fixed splits (from baseline OBJ1) ===
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))
print("Columns:", list(train_df.columns))

# Required columns:
req_cols = {"text_clean", "label"}
missing = req_cols - set(train_df.columns)
if missing:
    raise KeyError(f"Missing required columns in split CSV: {missing}. Found: {list(train_df.columns)}")

# For display/export
raw_col = "text" if "text" in test_df.columns else "text_clean"


Train/Val/Test: 960000 160000 480000
Columns: ['text_clean', 'label']


In [5]:
# === 3) Tokenization + Padding ===
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text_clean"].astype(str).tolist())

train_seq = tokenizer.texts_to_sequences(train_df["text_clean"].astype(str).tolist())
val_seq   = tokenizer.texts_to_sequences(val_df["text_clean"].astype(str).tolist())
test_seq  = tokenizer.texts_to_sequences(test_df["text_clean"].astype(str).tolist())

X_train = pad_sequences(train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val   = pad_sequences(val_seq,   maxlen=MAX_LEN, padding="post", truncating="post")
X_test  = pad_sequences(test_seq,  maxlen=MAX_LEN, padding="post", truncating="post")

y_train = train_df["label"].astype(int).to_numpy()
y_val   = val_df["label"].astype(int).to_numpy()
y_test  = test_df["label"].astype(int).to_numpy()

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)
print("Vocab size:", vocab_size)
print("X_train shape:", X_train.shape)


Vocab size: 50000
X_train shape: (960000, 50)


In [6]:
# === 4) Load GloVe vectors ===
if not GLOVE_PATH.exists():
    raise FileNotFoundError(
        f"Missing GloVe file at: {GLOVE_PATH}\n"
        "Download glove.6B.zip from https://nlp.stanford.edu/projects/glove/ and extract glove.6B.300d.txt\n"
        "Then place it under ./glove/glove.6B.300d.txt"
    )

embeddings_index = {}
with open(GLOVE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip().split(" ")
        word = parts[0]
        vec = np.asarray(parts[1:], dtype="float32")
        if vec.shape[0] != EMBED_DIM:
            continue
        embeddings_index[word] = vec

print("Loaded GloVe vectors:", len(embeddings_index))


Loaded GloVe vectors: 400000


In [7]:
# === 5) Build embedding matrix aligned with tokenizer vocab ===
rng = np.random.default_rng(RANDOM_STATE)
embedding_matrix = rng.normal(0, 0.05, size=(vocab_size, EMBED_DIM)).astype("float32")

hits = 0
for word, idx in tokenizer.word_index.items():
    if idx >= vocab_size:
        continue
    vec = embeddings_index.get(word)
    if vec is not None:
        embedding_matrix[idx] = vec
        hits += 1

print(f"Embedding coverage: {hits}/{vocab_size} = {hits/vocab_size:.2%}")


Embedding coverage: 37836/50000 = 75.67%


In [8]:
# === 6) Dataset & DataLoaders ===
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TextDataset(X_val, y_val),     batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TextDataset(X_test, y_test),   batch_size=BATCH_SIZE, shuffle=False)


In [9]:
# === 7) Define BiLSTM model ===
class BiLSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix: np.ndarray, hidden_dim: int = 128, dropout: float = 0.5):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))
        self.embedding.weight.requires_grad = True  # fine-tune embeddings

        self.spatial_dropout = nn.Dropout(0.2)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)                  # (B, T, D)
        x = self.spatial_dropout(x)
        _, (h_n, _) = self.lstm(x)             # h_n: (num_layers*2, B, H)
        h = torch.cat((h_n[-2], h_n[-1]), dim=1)  # (B, 2H)
        h = self.dropout(h)
        logits = self.fc(h).squeeze(1)         # (B,)
        return logits

model = BiLSTMClassifier(embedding_matrix, hidden_dim=HIDDEN_DIM).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(model)


BiLSTMClassifier(
  (embedding): Embedding(50000, 300)
  (spatial_dropout): Dropout(p=0.2, inplace=False)
  (lstm): LSTM(300, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)


In [ ]:
# === 8) Train loop (with best-val checkpoint) ===
def predict_probs(loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(device)
            logits = model(Xb)
            pb = torch.sigmoid(logits).cpu().numpy()
            probs.append(pb)
            labels.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(labels)

best_val_loss = float("inf")
best_path = MODEL_DIR / "lstm_model_binary.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for Xb, yb in train_loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / max(1, len(train_loader))

    # validation loss
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb = Xb.to(device)
            yb = yb.to(device)
            logits = model(Xb)
            loss = criterion(logits, yb)
            val_loss += loss.item()
    val_loss /= max(1, len(val_loader))

    print(f"Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_path)

print("Best val loss:", best_val_loss)
print("Saved best checkpoint to:", best_path.resolve())


Epoch 1/5 | train_loss=0.4080 | val_loss=0.3751
Epoch 2/5 | train_loss=0.3572 | val_loss=0.3727
Epoch 3/5 | train_loss=0.3289 | val_loss=0.3773
Epoch 4/5 | train_loss=0.3048 | val_loss=0.3913
Epoch 5/5 | train_loss=0.2848 | val_loss=0.4058
Best val loss: 0.3727463586330414
Saved best checkpoint to: /Users/victor/Desktop/Graduation_Thesis/Code/artifacts/lstm_model_binary.pt


In [11]:
# === 9) Load best checkpoint & evaluate on test (binary) ===
model.load_state_dict(torch.load(best_path, map_location=device))

test_probs, test_labels = predict_probs(test_loader)
pred_binary = (test_probs >= 0.5).astype(int)

print(classification_report(test_labels, pred_binary, digits=4))
print("Confusion matrix:\n", confusion_matrix(test_labels, pred_binary))


              precision    recall  f1-score   support

         0.0     0.8244    0.8487    0.8363    240000
         1.0     0.8441    0.8192    0.8315    240000

    accuracy                         0.8339    480000
   macro avg     0.8342    0.8339    0.8339    480000
weighted avg     0.8342    0.8339    0.8339    480000

Confusion matrix:
 [[203677  36323]
 [ 43386 196614]]


In [12]:
# === 10) Neutral post-hoc mapping ===
def proba_to_3class(probs: np.ndarray, neg_th: float = 0.4, pos_th: float = 0.6):
    probs = np.asarray(probs, dtype=float)
    out = np.empty(probs.shape[0], dtype=object)
    out[probs <= neg_th] = "NEGATIVE"
    out[probs >= pos_th] = "POSITIVE"
    mid = (probs > neg_th) & (probs < pos_th)
    out[mid] = "NEUTRAL"
    return out

pred_3class = proba_to_3class(test_probs, neg_th=NEG_TH, pos_th=POS_TH)
neutral_rate = (pred_3class == "NEUTRAL").mean()
print(f"Neutral rate on test: {neutral_rate:.4f} (NEG_TH={NEG_TH}, POS_TH={POS_TH})")


Neutral rate on test: 0.1075 (NEG_TH=0.4, POS_TH=0.6)


In [ ]:
# === 11) Export artifacts (same style as baseline) ===
threshold_path = CONFIG_DIR / "neutral_thresholds.json"
with open(threshold_path, "w", encoding="utf-8") as f:
    json.dump(
        {"neg_th": float(NEG_TH), "pos_th": float(POS_TH), "policy": "post_hoc_thresholding_on_p_positive"},
        f,
        ensure_ascii=False,
        indent=2,
    )

pred_df = pd.DataFrame({
    "text": test_df[raw_col].astype(str).tolist(),
    "text_clean": test_df["text_clean"].astype(str).tolist(),
    "y_true": test_labels.astype(int),
    "proba_pos": test_probs.astype(float),
    "pred_binary": pred_binary.astype(int),
})

binary_path = PRED_DIR / "pred_test_lstm_binary.csv"
pred_df.to_csv(binary_path, index=False, encoding="utf-8")

neutral_df = pred_df.copy()
neutral_df["pred_3class"] = pred_3class
neutral_path = PRED_DIR / f"pred_test_lstm_neutral_{NEG_TH:.2f}_{POS_TH:.2f}.csv"
neutral_df.to_csv(neutral_path, index=False, encoding="utf-8")

print("Saved thresholds:", threshold_path.resolve())
print("Saved binary preds:", binary_path.resolve())
print("Saved neutral preds:", neutral_path.resolve())


Saved thresholds: /Users/victor/Desktop/Graduation_Thesis/Code/artifacts/config/neutral_thresholds.json
Saved binary preds: /Users/victor/Desktop/Graduation_Thesis/Code/artifacts/predictions/pred_test_lstm_binary.csv
Saved neutral preds: /Users/victor/Desktop/Graduation_Thesis/Code/artifacts/predictions/pred_test_lstm_neutral_0.40_0.60.csv
